In [218]:
import pandas as pd
import numpy as np
from pathlib import Path
import csv
import geopandas as gpd
import folium

In [219]:
## Set up file paths
## project root is two levels above notebooks/raw_data_processing
PROJECT_ROOT = Path.cwd().parent.parent
SAVE_PATH = PROJECT_ROOT / 'data' / 'final_dataset' / 'main_dataset.csv'

SHAPEFILE = PROJECT_ROOT / 'data' / 'raw' / 'fsa_boundary' / 'gfsa000a11a_e.shp'
RADON_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'radon-concentration-cleaned.csv'
FSA_BOUNDARY_CENTROIDS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'fsa_centroids.csv'
CENSUS_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'fsa_census.csv'
GEO_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'geology_fsa_radon.csv'
SURFICIAL_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'surficial_fsa_radon.csv'
URANIUM_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'fsa_uranium.csv'
WEATHER_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'average_heating_days_cleaned.csv'

FSA_MAP_HTML_PATH = PROJECT_ROOT / 'figures' / 'maps' / 'canada_fsa_map.html'

Radon data availability

In [220]:
radon_df = pd.read_csv(RADON_DATA_PATH)
radon_df.info()
print(f"Unique FSA count in radon data: {radon_df['FSA'].nunique()}")
radon_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13815 entries, 0 to 13814
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   FSA            13814 non-null  object 
 1   n_days         13814 non-null  float64
 2   concentration  13814 non-null  float64
dtypes: float64(2), object(1)
memory usage: 323.9+ KB
Unique FSA count in radon data: 1015


,FSA,n_days,concentration
0,A0A,127.0,20.0
1,A0A,108.0,36.0
2,A0E,91.0,7.5
3,A0A,91.0,31.0
4,A0C,98.0,26.0


**Note:** Need to keep track of the intersection of FSAs for which we have data for all the features and the final target. The list `relavant_fsas` would keep track of this.

In [221]:
relevant_fsas = set(radon_df['FSA'].dropna().unique())
print(f"Unique FSA count in census data: {len(relevant_fsas)}")

Unique FSA count in census data: 1015


### FSA boundary data processing

In [222]:
fsa_center_df = pd.read_csv(FSA_BOUNDARY_CENTROIDS_PATH)
fsa_center_df.info()
fsa_center_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1621 entries, 0 to 1620
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   FSA        1621 non-null   object 
 1   longitude  1621 non-null   float64
 2   latitude   1621 non-null   float64
dtypes: float64(2), object(1)
memory usage: 38.1+ KB


,FSA,longitude,latitude
0,A0A,-53.087800,47.317160
1,A0B,-53.691063,47.342621
2,A0C,-53.725413,48.375277
3,A0E,-54.777629,47.384855
4,A0G,-54.457730,49.130629


### Check and update relevant FSAs

In [223]:
fsa_center_relevant_df = fsa_center_df[fsa_center_df['FSA'].isin(relevant_fsas)]
print(f"Unique and relevant FSA count in center data: {fsa_center_relevant_df['FSA'].nunique()}")
relevant_fsas = set(fsa_center_relevant_df['FSA'].unique())

Unique and relevant FSA count in center data: 1014


### Census data processing

In [224]:
census_df = pd.read_csv(CENSUS_DATA_PATH)
census_df.info()
census_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1641 entries, 0 to 1640
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   FSA                      1641 non-null   object 
 1   pct_single_detached      1620 non-null   float64
 2   pct_highrise             1620 non-null   float64
 3   pct_other_attached       1620 non-null   float64
 4   pct_movable              1620 non-null   float64
 5   avg_rooms                1619 non-null   float64
 6   pct_major_repair         1619 non-null   float64
 7   pct_pre_1980             1619 non-null   float64
 8   pct_1981_2000            1619 non-null   float64
 9   pct_post_2001            1619 non-null   float64
 10  median_home_value        1611 non-null   float64
 11  pop_2016                 1641 non-null   float64
 12  num_dwellings            1641 non-null   float64
 13  occ_dwellings            1641 non-null   float64
 14  median_age              

,FSA,pct_single_detached,pct_highrise,pct_other_attached,pct_movable,avg_rooms,pct_major_repair,pct_pre_1980,pct_1981_2000,pct_post_2001,...,pct_low_income,med_income,pct_high_income,pct_govt_transfers,unemployment_rate,pct_non_laborer,pct_bachelor_plus,pct_overcrowded,pct_unsuitable_housing,pct_housing_poor_owners
0,A0A,0.923057,0.000257,0.071024,0.005661,7.0,0.077341,0.531298,0.268753,0.199948,...,17.0,63866.0,2.541145,20.7,16.4,45.566219,10.614192,0.284532,2.069322,11.6
1,A0B,0.947727,0.000000,0.039773,0.012500,6.9,0.078798,0.575156,0.285876,0.138968,...,16.5,59154.0,2.305646,23.7,18.5,48.388936,7.300989,0.283447,1.133787,9.5
2,A0C,0.933988,0.000000,0.052632,0.013381,6.6,0.065825,0.591524,0.286745,0.121731,...,20.5,52112.0,0.942507,29.8,26.8,47.597043,6.049563,0.000000,0.991885,8.9
3,A0E,0.933854,0.000521,0.055208,0.010417,6.9,0.072320,0.661290,0.241935,0.096774,...,17.1,60954.0,2.530283,21.7,23.5,48.757764,7.221542,0.156087,1.457574,7.9
4,A0G,0.936184,0.000000,0.057237,0.006579,6.9,0.087558,0.576505,0.297137,0.126357,...,20.7,53342.0,1.295602,28.6,27.5,50.771504,6.316348,0.362080,1.283739,8.2


In [225]:
census_relevant_df = census_df[census_df['FSA'].isin(relevant_fsas)]
print(f"Unique FSA count in census data: {census_relevant_df['FSA'].nunique()}")
relevant_fsas = set(census_relevant_df['FSA'].unique())

Unique FSA count in census data: 1014


### Geological data processing

In [226]:
geo_df = pd.read_csv(GEO_DATA_PATH)
geo_surficial_df = pd.read_csv(SURFICIAL_DATA_PATH)
geo_df.info()
geo_surficial_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13567 entries, 0 to 13566
Data columns (total 29 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ProvinceTerritory                    13567 non-null  object 
 1   FSA                                  13567 non-null  object 
 2   TestDurationInDays                   13567 non-null  float64
 3   AverageRadonConcentrationInBqPerM3   13567 non-null  float64
 4   RXTP_intrusive rocks                 13567 non-null  float64
 5   RXTP_metamorphic rocks               13567 non-null  float64
 6   RXTP_sedimentary and volcanic rocks  13567 non-null  float64
 7   RXTP_sedimentary rocks               13567 non-null  float64
 8   RXTP_unknown                         13567 non-null  float64
 9   RXTP_volcanic rocks                  13567 non-null  float64
 10  GEOLPROV_Appalachian Orogen          13567 non-null  float64
 11  GEOLPROV_Arctic Continental 

In [227]:
## Remove radon test specific columns and drop duplicates to ensure one row per FSA in geological datasets
geo_df.drop(columns=['TestDurationInDays', 'AverageRadonConcentrationInBqPerM3'], inplace=True)
geo_df.drop_duplicates(inplace=True)
print(f"Shape of geo_df after processing: {geo_df.shape}")
geo_surficial_df.drop(columns=['TestDurationInDays', 'AverageRadonConcentrationInBqPerM3'], inplace=True)
geo_surficial_df.drop_duplicates(inplace=True)
print(f"Shape of geo_surficial_df after processing: {geo_surficial_df.shape}")

Shape of geo_df after processing: (1014, 27)
Shape of geo_surficial_df after processing: (1014, 16)


In [228]:
print(f"Unique FSA count in geological data: {geo_df['FSA'].nunique()}")
print(f"Unique FSA count in surficial geological data: {geo_surficial_df['FSA'].nunique()}")
print(set(geo_df['FSA'].unique()) == relevant_fsas)
print(set(geo_surficial_df['FSA'].unique()) == relevant_fsas)

Unique FSA count in geological data: 1014
Unique FSA count in surficial geological data: 1014
True
True


### Uranium data processing

In [229]:
uranium_df = pd.read_csv(URANIUM_DATA_PATH)
uranium_df.info()
print(f"Unique FSA count in uranium data: {uranium_df['FSA'].nunique()}")
print(set(uranium_df['FSA'].unique()) == relevant_fsas)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1621 entries, 0 to 1620
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   FSA           1621 non-null   object 
 1   mean_uranium  945 non-null    float64
 2   max_uranium   945 non-null    float64
dtypes: float64(2), object(1)
memory usage: 38.1+ KB
Unique FSA count in uranium data: 1621
False


In [230]:
uranium_df = uranium_df[uranium_df['FSA'].isin(relevant_fsas)]
print(f"Unique FSA count in filtered uranium data: {uranium_df['FSA'].nunique()}")
relevant_fsas = set(uranium_df['FSA'].unique())
uranium_df.info()

Unique FSA count in filtered uranium data: 1014
<class 'pandas.core.frame.DataFrame'>
Index: 1014 entries, 0 to 1620
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   FSA           1014 non-null   object 
 1   mean_uranium  666 non-null    float64
 2   max_uranium   666 non-null    float64
dtypes: float64(2), object(1)
memory usage: 31.7+ KB


### Weather data processing

In [231]:
average_heating_days_df = pd.read_csv(WEATHER_DATA_PATH)
average_heating_days_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   FSA                   1014 non-null   object 
 1   average_heating_days  1014 non-null   float64
dtypes: float64(1), object(1)
memory usage: 16.0+ KB


In [232]:
print(set(average_heating_days_df['FSA'].unique()) == relevant_fsas)

True


Finally merge with the radon concentration data

In [233]:
## Filter all datasets to only include relevant FSAs (just a sanity check)
radon_relevant_df = radon_df[radon_df['FSA'].isin(relevant_fsas)]
fsa_center_relevant_df = fsa_center_df[fsa_center_df['FSA'].isin(relevant_fsas)]
census_relevant_df = census_df[census_df['FSA'].isin(relevant_fsas)]
geo_relevant_df = geo_df[geo_df['FSA'].isin(relevant_fsas)]
geo_surficial_relevant_df = geo_surficial_df[geo_surficial_df['FSA'].isin(relevant_fsas)]
average_heating_days_relevant_df = average_heating_days_df[average_heating_days_df['FSA'].isin(relevant_fsas)]
uranium_relevant_df = uranium_df[uranium_df['FSA'].isin(relevant_fsas)]

In [234]:
print(f"Shape of radon data: {radon_relevant_df.shape}")
print(f"Shape of FSA center data: {fsa_center_relevant_df.shape}")
print(f"Shape of census data: {census_relevant_df.shape}")
print(f"Shape of geological data: {geo_relevant_df.shape}")
print(f"Shape of surficial geological data: {geo_surficial_relevant_df.shape}")
print(f"Shape of average heating days data: {average_heating_days_relevant_df.shape}")
print(f"Shape of uranium data: {uranium_relevant_df.shape}")

Shape of radon data: (13567, 3)
Shape of FSA center data: (1014, 3)
Shape of census data: (1014, 29)
Shape of geological data: (1014, 27)
Shape of surficial geological data: (1014, 16)
Shape of average heating days data: (1014, 2)
Shape of uranium data: (1014, 3)


In [235]:
set(radon_relevant_df['FSA'].unique()) == set(fsa_center_relevant_df['FSA'].unique()) == set(census_relevant_df['FSA'].unique()) == set(geo_relevant_df['FSA'].unique()) == set(geo_surficial_relevant_df['FSA'].unique()) == set(average_heating_days_relevant_df['FSA'].unique()) #== set(uranium_relevant_df['FSA'].unique())

True

In [236]:
## Merge all datasets on FSA using left joins to ensure we keep all FSA entries including multiple radon 
## test entries per FSA from the radon dataset
merged_df = pd.merge(radon_relevant_df, fsa_center_relevant_df, on='FSA', how='left')
print(f"Shape after merging radon and FSA center data: {merged_df.shape}")
merged_df = pd.merge(merged_df, census_relevant_df, on='FSA', how='left')
print(f"Shape after merging with census data: {merged_df.shape}")
merged_df = pd.merge(merged_df, geo_relevant_df, on='FSA', how='left')
print(f"Shape after merging with geological data: {merged_df.shape}")
merged_df = pd.merge(merged_df, geo_surficial_relevant_df, on='FSA', how='left')
print(f"Shape after merging with surficial geological data: {merged_df.shape}")
merged_df = pd.merge(merged_df, average_heating_days_relevant_df, on='FSA', how='left')
print(f"Shape after merging with average heating days data: {merged_df.shape}")
## Dropping nan values for all other features except Uranium data
merged_df =merged_df.dropna()

## Merging all uranium rows for each relevant FSA including nan values
merged_df = pd.merge(merged_df, uranium_relevant_df, on='FSA', how='left')
print(f"Shape after merging with uranium data: {merged_df.shape}")

Shape after merging radon and FSA center data: (13567, 5)
Shape after merging with census data: (13567, 33)
Shape after merging with geological data: (13567, 59)
Shape after merging with surficial geological data: (13567, 74)
Shape after merging with average heating days data: (13567, 75)
Shape after merging with uranium data: (13077, 77)


In [237]:
merged_df.head()

,FSA,n_days,concentration,longitude,latitude,pct_single_detached,pct_highrise,pct_other_attached,pct_movable,avg_rooms,...,Glaciomarine sediments,Lacustrine sediments,Marine sediments,Organic deposits,Volcanic deposits,Weathered bedrock or regolith,ProvinceTerritory_y,average_heating_days,mean_uranium,max_uranium
0,A0A,127.0,20.0,-53.087800,47.317160,0.923057,0.000257,0.071024,0.005661,7.0,...,0.000000,0.0,0.030408,0.0,0.0,0.0,NL,305.532585,NaN,NaN
1,A0A,108.0,36.0,-53.087800,47.317160,0.923057,0.000257,0.071024,0.005661,7.0,...,0.000000,0.0,0.030408,0.0,0.0,0.0,NL,305.532585,NaN,NaN
2,A0E,91.0,7.5,-54.777629,47.384855,0.933854,0.000521,0.055208,0.010417,6.9,...,0.122135,0.0,0.125665,0.0,0.0,0.0,NL,326.720975,0.637329,3.509083
3,A0A,91.0,31.0,-53.087800,47.317160,0.923057,0.000257,0.071024,0.005661,7.0,...,0.000000,0.0,0.030408,0.0,0.0,0.0,NL,305.532585,NaN,NaN
4,A0C,98.0,26.0,-53.725413,48.375277,0.933988,0.000000,0.052632,0.013381,6.6,...,0.000000,0.0,0.000000,0.0,0.0,0.0,NL,293.239321,0.846531,2.030009


In [238]:
## Copy the merged dataframe to a new variable and save to CSV
main_df = merged_df.copy()
main_df.columns

Index(['FSA', 'n_days', 'concentration', 'longitude', 'latitude',
       'pct_single_detached', 'pct_highrise', 'pct_other_attached',
       'pct_movable', 'avg_rooms', 'pct_major_repair', 'pct_pre_1980',
       'pct_1981_2000', 'pct_post_2001', 'median_home_value', 'pop_2016',
       'num_dwellings', 'occ_dwellings', 'median_age', 'avg_household_size',
       'pct_owned', 'pct_rented', 'pct_band', 'pct_low_income', 'med_income',
       'pct_high_income', 'pct_govt_transfers', 'unemployment_rate',
       'pct_non_laborer', 'pct_bachelor_plus', 'pct_overcrowded',
       'pct_unsuitable_housing', 'pct_housing_poor_owners',
       'ProvinceTerritory_x', 'RXTP_intrusive rocks', 'RXTP_metamorphic rocks',
       'RXTP_sedimentary and volcanic rocks', 'RXTP_sedimentary rocks',
       'RXTP_unknown', 'RXTP_volcanic rocks', 'GEOLPROV_Appalachian Orogen',
       'GEOLPROV_Arctic Continental Shelf', 'GEOLPROV_Arctic Platform',
       'GEOLPROV_Atlantic Continental Shelf', 'GEOLPROV_Bear Province'

In [239]:
main_df['ProvinceTerritory'] = main_df['ProvinceTerritory_x']
main_df.drop(columns=['ProvinceTerritory_x', 'ProvinceTerritory_y'], inplace=True)
print(f"Final dataset shape: {main_df.shape}")
main_df.to_csv(SAVE_PATH, index=False)

Final dataset shape: (13077, 76)


In [240]:
print(f"We have non-null value for all features (except Uranium) for a total of {main_df['FSA'].nunique()} FSAs")
print(f"We have non-null value  Uranium for a total of {main_df['FSA'].nunique()-uranium_relevant_df['mean_uranium'].isna().sum()} FSAs")

We have non-null value for all features (except Uranium) for a total of 1005 FSAs
We have non-null value  Uranium for a total of 657 FSAs


### Create a map with various data as pop-up

In [241]:
census_columns = census_df.columns.tolist()[1:]  # Remove FSA column for clarity
geo_columns = geo_df.columns.tolist()[2:]  
geo_surficial_columns = geo_surficial_df.columns.tolist()[1:-1]  
print("Census columns:", census_columns)
print("Geological columns:", geo_columns)
print("Surficial geological columns:", geo_surficial_columns)

Census columns: ['pct_single_detached', 'pct_highrise', 'pct_other_attached', 'pct_movable', 'avg_rooms', 'pct_major_repair', 'pct_pre_1980', 'pct_1981_2000', 'pct_post_2001', 'median_home_value', 'pop_2016', 'num_dwellings', 'occ_dwellings', 'median_age', 'avg_household_size', 'pct_owned', 'pct_rented', 'pct_band', 'pct_low_income', 'med_income', 'pct_high_income', 'pct_govt_transfers', 'unemployment_rate', 'pct_non_laborer', 'pct_bachelor_plus', 'pct_overcrowded', 'pct_unsuitable_housing', 'pct_housing_poor_owners']
Geological columns: ['RXTP_intrusive rocks', 'RXTP_metamorphic rocks', 'RXTP_sedimentary and volcanic rocks', 'RXTP_sedimentary rocks', 'RXTP_unknown', 'RXTP_volcanic rocks', 'GEOLPROV_Appalachian Orogen', 'GEOLPROV_Arctic Continental Shelf', 'GEOLPROV_Arctic Platform', 'GEOLPROV_Atlantic Continental Shelf', 'GEOLPROV_Bear Province', 'GEOLPROV_Churchill Province', 'GEOLPROV_Cordilleran Orogen', 'GEOLPROV_Grenville Province', 'GEOLPROV_Hudson Bay Lowlands', 'GEOLPROV_Innui

In [246]:
fsa_gpd = gpd.read_file(SHAPEFILE)
fsa_gpd.rename(columns={'CFSAUID': 'FSA'}, inplace=True)

## adding radon data
fsa_gpd['radon_concentration_mean'] = fsa_gpd['FSA'].map(
    main_df.groupby('FSA')['concentration'].mean())
fsa_gpd['radon_concentration_max'] = fsa_gpd['FSA'].map(
    main_df.groupby('FSA')['concentration'].max())

## adding centroid coordinate
fsa_gpd = pd.merge(fsa_gpd, fsa_center_df, on='FSA', how='left')

## adding population data
fsa_gpd = pd.merge(fsa_gpd, census_df[['FSA', 'pop_2016']], on='FSA', how = 'left')

## adding average heating days and mean uranium data
fsa_gpd = pd.merge(fsa_gpd, average_heating_days_df[['FSA', 'average_heating_days']], on='FSA', how='left')
fsa_gpd = pd.merge(fsa_gpd, uranium_df[['FSA', 'mean_uranium']], on='FSA', how='left')

## adding dominant geological data
geo_df['dominant_geology'] = geo_df[geo_columns].idxmax(axis=1)
fsa_gpd = pd.merge(fsa_gpd, geo_df[['FSA', 'dominant_geology']], on= 'FSA', how= 'left')
geo_surficial_df['dominant_surficial_geology'] = geo_surficial_df[geo_surficial_columns].idxmax(axis=1)
fsa_gpd = pd.merge(fsa_gpd, geo_surficial_df[['FSA', 'dominant_surficial_geology']], on='FSA', how= 'left')


fsa_gpd.head()

,FSA,PRUID,PRNAME,geometry,radon_concentration_mean,radon_concentration_max,longitude,latitude,pop_2016,average_heating_days,mean_uranium,dominant_geology,dominant_surficial_geology
0,A0A,10,Newfoundland and Labrador / Terre-Neuve-et-Lab...,"MULTIPOLYGON (((-53.27283 47.37429, -53.27288 ...",69.230769,444.0,-53.087800,47.317160,46587.0,305.532585,NaN,GEOLPROV_Appalachian Orogen,Glacial sediments
1,A0B,10,Newfoundland and Labrador / Terre-Neuve-et-Lab...,"MULTIPOLYGON (((-53.27864 47.37389, -53.27771 ...",48.833333,122.0,-53.691063,47.342621,19792.0,311.329409,NaN,GEOLPROV_Appalachian Orogen,Glacial sediments
2,A0C,10,Newfoundland and Labrador / Terre-Neuve-et-Lab...,"POLYGON ((-52.99698 48.68359, -52.99591 48.655...",53.900000,140.0,-53.725413,48.375277,12587.0,293.239321,0.846531,GEOLPROV_Appalachian Orogen,Glacial sediments
3,A0E,10,Newfoundland and Labrador / Terre-Neuve-et-Lab...,"POLYGON ((-54.42293 48.19427, -54.41571 48.186...",31.588235,181.0,-54.777629,47.384855,22294.0,326.720975,0.637329,GEOLPROV_Appalachian Orogen,Glacial sediments
4,A0G,10,Newfoundland and Labrador / Terre-Neuve-et-Lab...,"POLYGON ((-53.16336 49.41443, -53.19745 49.328...",70.769231,1548.0,-54.457730,49.130629,35266.0,286.243154,1.047329,GEOLPROV_Appalachian Orogen,Glacial sediments


In [247]:
fsa_gpd.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1621 entries, 0 to 1620
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   FSA                         1621 non-null   object  
 1   PRUID                       1621 non-null   object  
 2   PRNAME                      1621 non-null   object  
 3   geometry                    1621 non-null   geometry
 4   radon_concentration_mean    1005 non-null   float64 
 5   radon_concentration_max     1005 non-null   float64 
 6   longitude                   1621 non-null   float64 
 7   latitude                    1621 non-null   float64 
 8   pop_2016                    1620 non-null   float64 
 9   average_heating_days        1014 non-null   float64 
 10  mean_uranium                666 non-null    float64 
 11  dominant_geology            1014 non-null   object  
 12  dominant_surficial_geology  1014 non-null   object  
dtypes: float64

In [245]:
import folium
from folium import GeoJson

# Create base map centered on Canada
m = folium.Map(location=[56, -106], zoom_start=4)

# Add FSA boundaries with popup showing FSA code and other info
for idx, row in fsa_gpd.iterrows():
    # Create popup text with desired information
    popup_text = f"""<b>FSA Code:</b> {row['FSA']}<br>
                     <b>Province/Territory:</b> {row['PRNAME']} <br>
                     <b>Centroid coordinate (long,lat):</b> ({row['longitude']:.2f}, {row['latitude']:.2f})<br>
                     <b>Average Radon Concentration (Bq/m3):</b> {row['radon_concentration_mean']:.2f}<br>
                     <b>Max Radon Concentration (Bq/m3):</b> {row['radon_concentration_max']:.2f}<br>
                     <b>Population (2016):</b> {row['pop_2016']:.0f} <br>
                     <b>Average Heating Days:</b> {row['average_heating_days']:.1f} <br>
                     <b>Mean Uranium Concentration:</b> {row['mean_uranium']:.2f} <br>
                     <b>Dominant Geology:</b> {row['dominant_geology']} <br>
                     <b>Dominant Surficial Geology:</b> {row['dominant_surficial_geology']} <br>
                     """
    
    folium.GeoJson(
        row['geometry'],
        popup=folium.Popup(popup_text, max_width=300),
        style_function=lambda x: {'color': 'blue', 'fillOpacity': 0.1}
    ).add_to(m)

m.save(FSA_MAP_HTML_PATH)  # Opens in browser